# Hands-on Part 2: Network measures with NetworkX 🕸️

*CSS 2026 · Yangliu Fan · Freie Universität Berlin*

In Part 1 you learned lists, tuples and pandas tables. Now we use them to analyse networks. In this notebook you will:

1. learn what a **function** is and write your own,
2. build networks with the `networkx` library,
3. compute the measures from the lectures: **degree**, **degree distribution**, **paths and distance**, **diameter**, **betweenness centrality** and **eigenvector centrality**,
4. apply everything to a real network.

As before: run cells from top to bottom with **Shift + Enter**. Cells marked ⚠️ give an error on purpose.

## 0. Setup

We import the libraries we need. `networkx` is usually shortened to `nx`, and `matplotlib` draws the pictures. Colab already has all three installed.

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

## 1. Functions

A **function** takes some input, does something with it, and gives back an output. You have already used several:

- `print(...)` shows something
- `len(...)` counts the items in a list
- `sum(...)` and `max(...)` add up and find the largest number

The input goes between the round brackets and is called the **argument**.

In [ ]:
degrees = [4, 3, 4, 5, 2]

print(len(degrees))
print(sum(degrees))
print(max(degrees))

### Writing your own function

In Part 1 we calculated the density by typing the formula every time. With a function, we write the formula **once** and reuse it for any network:

In [ ]:
def density(N, L):
    L_max = N * (N - 1) / 2
    return L / L_max

Let's go through each part:

- `def` tells Python we are **def**ining a function.
- `density` is the name we choose for it.
- `(N, L)` are the **arguments**: the inputs the function needs.
- `:` and the **indented** lines below form the body of the function. ⚠️ The indentation (4 spaces) is required.
- `return` gives the result back.

Running the cell above only *defines* the function; nothing happens yet. Now we **call** it:

In [ ]:
print(density(4, 4))       # the network from the Week 2 slides
print(density(25, 40))     # the seminar from Task 1 in Part 1
print(density(1000, 3000)) # a larger, sparse network

Here is a second function, for the average degree $\langle k \rangle = \dfrac{2L}{N}$:

In [ ]:
def average_degree(N, L):
    return 2 * L / N

average_degree(4, 4)

⚠️ **Error on purpose.** Here the line inside the function is not indented. *Read the error message: what is Python complaining about?*

In [ ]:
def max_links(N):
return N * (N - 1) / 2

**Task 1:** Fix the function above by indenting the second line, then use it to calculate the maximum number of links for a network of 50 nodes.

In [ ]:
# your code here

### Functions from libraries

Libraries are collections of functions written by other people. You call them with the library's nickname and a dot, for example `pd.DataFrame(...)` or `nx.density(...)`. Some functions belong to an object and come after its name, such as `G.number_of_nodes()`. These are called **methods**.

💡 In Colab, type `nx.` and wait a moment: a list of all available functions appears.

## 2. Your first network in NetworkX

Remember the edge list from the Week 2 slides: `{(1, 2), (1, 4), (3, 4), (2, 4)}`. We create an empty undirected network (a `Graph`) and add the edge list to it:

In [ ]:
edges = [(1, 2), (1, 4), (3, 4), (2, 4)]

G_small = nx.Graph()            # an empty undirected network
G_small.add_edges_from(edges)   # add the links (the nodes are added automatically)

In [ ]:
print("Nodes:", G_small.nodes)
print("Links:", G_small.edges)
print("N =", G_small.number_of_nodes())
print("L =", G_small.number_of_edges())

Let's draw it. `with_labels=True` shows the node names.

In [ ]:
nx.draw(G_small, with_labels=True, node_color="lightblue")

NetworkX has a function for density. Does it give the same result as our own function?

In [ ]:
print("Our function:", density(G_small.number_of_nodes(), G_small.number_of_edges()))
print("NetworkX:    ", nx.density(G_small))

🕸️ **Directed networks** (Week 1). For directed networks, use `nx.DiGraph()` instead. The order inside each tuple now matters: `(1, 2)` is an arrow *from* 1 *to* 2.

In [ ]:
D = nx.DiGraph()
D.add_edges_from(edges)
nx.draw(D, with_labels=True, node_color="lightblue", arrows=True)

In the rest of this notebook we work with undirected networks.

## 3. Our example: two friend groups

Here is the friendship network from Part 1: a group of friends on the left, another group on the right, and Felix, who knows people in both.

In [ ]:
friendships = [
    ("Ana", "Ben"), ("Ana", "Chen"), ("Ana", "Dara"), ("Ana", "Eva"),
    ("Ben", "Chen"), ("Ben", "Dara"), ("Chen", "Dara"), ("Dara", "Eva"),
    ("Dara", "Felix"), ("Chen", "Felix"), ("Felix", "Gus"), ("Felix", "Hana"),
    ("Gus", "Hana"), ("Gus", "Ivo"), ("Gus", "Jonas"), ("Gus", "Kim"),
    ("Gus", "Lena"), ("Hana", "Ivo"), ("Kim", "Lena"), ("Ivo", "Jonas"),
    ("Hana", "Kim"),
]

G = nx.Graph()
G.add_edges_from(friendships)

print("N =", G.number_of_nodes())
print("L =", G.number_of_edges())
print("Density:", nx.density(G))

To draw networks nicely, we wrote a small function for you. **You don't need to understand every line.** What matters is how to use it:

- `draw_network(G)` draws the network.
- `draw_network(G, measure)` makes nodes with a higher value of a measure **bigger**.

In [ ]:
def draw_network(G, measure=None, title=""):
    positions = nx.kamada_kawai_layout(G)   # places linked nodes close together, same layout every time
    if measure is None:
        sizes = 500
    else:
        biggest = max(measure.values())
        sizes = [200 + 1800 * measure[node] / biggest for node in G.nodes]
    plt.figure(figsize=(8, 5))
    nx.draw(G, positions, with_labels=True, node_size=sizes,
            node_color="lightblue", edge_color="gray", font_size=9)
    plt.title(title)
    plt.show()

draw_network(G, title="Two friend groups")

## 4. Degree

The **degree** $k$ of a node is the number of links it has (Week 3). In a friendship network: how many friends someone has.

In [ ]:
G.degree("Gus")

To get the degree of **every** node at once:

In [ ]:
degree = dict(G.degree())
degree

💡 **What are the curly brackets?** NetworkX gives the result as a **dictionary**: a collection of `name: value` pairs. You look up a value by putting the name in square brackets:

In [ ]:
degree["Felix"]

Dictionaries are hard to read, so let's put the result into a pandas table. We give `pd.DataFrame` the column name and the dictionary, and pandas uses the node names as row labels.

In [ ]:
table = pd.DataFrame({"degree": degree})
table.sort_values("degree", ascending=False)

**Average degree.** Let's check that the mean of the column equals our formula $2L/N$:

In [ ]:
print("Mean of the degree column:", table["degree"].mean())
print("Our function 2L/N:        ", average_degree(G.number_of_nodes(), G.number_of_edges()))

**Degree centrality** is the degree divided by the highest possible degree, $N-1$. It lets us compare networks of different sizes (a value of 1 means "connected to everyone").

In [ ]:
table["degree_centrality"] = pd.Series(nx.degree_centrality(G))
table.sort_values("degree", ascending=False)

In [ ]:
draw_network(G, degree, title="Node size = degree")

**Task 2:** Who has the highest degree? Recall the *limitations of degree centrality* from Week 3: is this person also the most important for holding the network together?

**Your answer:**

## 5. Degree distribution

The **degree distribution** $P(k)$ tells us the probability that a randomly chosen node has degree $k$ (Week 3).

**Step 1:** count how many nodes have each degree:

In [ ]:
counts = table["degree"].value_counts().sort_index()
counts

**Step 2:** divide by the number of nodes $N$ to turn counts into probabilities. The probabilities always add up to 1.

In [ ]:
P_k = counts / G.number_of_nodes()
print(P_k)
print("Sum:", P_k.sum())

**Step 3:** plot it as a bar chart:

In [ ]:
P_k.plot(kind="bar", color="steelblue")
plt.xlabel("Degree k")
plt.ylabel("P(k)")
plt.title("Degree distribution")
plt.show()

With only 12 people, the distribution is not very informative. Let's look at a larger, real network. NetworkX comes with **Zachary's karate club**: friendships among 34 members of a university karate club in the 1970s.

In [ ]:
karate = nx.karate_club_graph()
print("N =", karate.number_of_nodes())
print("L =", karate.number_of_edges())

karate_degree = pd.Series(dict(karate.degree()))
P_k_karate = karate_degree.value_counts().sort_index() / karate.number_of_nodes()

P_k_karate.plot(kind="bar", color="steelblue")
plt.xlabel("Degree k")
plt.ylabel("P(k)")
plt.title("Degree distribution: karate club")
plt.show()

**Task 3:** Describe the shape of the karate club distribution. Do most members have few or many friends? Remember the friendship paradox from Week 2: why does this shape mean that your friends have, on average, more friends than you?

**Your answer:**

## 6. Paths and distance

A **path** is a sequence of nodes in which each node is linked to the next one. The **shortest path** between two nodes is the path with the fewest links, and its length is the **distance** (Week 2).

Is there a path from Lena to Eva at all?

In [ ]:
nx.has_path(G, "Lena", "Eva")

Which nodes are on the shortest path? The result is a list of nodes:

In [ ]:
path = nx.shortest_path(G, "Lena", "Eva")
path

The distance is the number of **links** on the path, which is the number of nodes minus 1:

In [ ]:
print("Length from the path:", len(path) - 1)
print("NetworkX distance:   ", nx.shortest_path_length(G, "Lena", "Eva"))

🕸️ **Disconnected nodes.** If there is no path between two nodes, their distance is infinite. Let's add a new student, Zoe, who has no friends in the network yet. We work on a copy so that `G` stays unchanged.

In [ ]:
G_with_zoe = G.copy()
G_with_zoe.add_node("Zoe")

nx.has_path(G_with_zoe, "Ana", "Zoe")

⚠️ **Error on purpose.** Asking for the shortest path to Zoe gives an error, because no path exists:

In [ ]:
nx.shortest_path_length(G_with_zoe, "Ana", "Zoe")

**Task 4:** Find the shortest path between **Ben** and **Jonas**, and its length. Who are the people a message would have to pass through?

In [ ]:
# your code here

## 7. Diameter

The **diameter** is the longest shortest path in the network: the largest distance between any pair of nodes (Week 2).

In [ ]:
nx.diameter(G)

The path from Lena to Eva that we found above has length 4, the same as the diameter: Lena and Eva are as far apart as any two people in this network.

⚠️ The diameter only works for **connected** networks. For `G_with_zoe` the diameter would be infinite, and NetworkX would give an error. You can check first with `nx.is_connected(G)`.

In [ ]:
print(nx.is_connected(G))
print(nx.is_connected(G_with_zoe))

## 8. Betweenness centrality

**Betweenness centrality** asks: how many pairs of people have to go through you to reach one another along the shortest path? (Week 3)

Nodes with high betweenness are **bridges** between groups: they can control the flow of information.

In [ ]:
betweenness = nx.betweenness_centrality(G)
table["betweenness"] = pd.Series(betweenness)
table.sort_values("betweenness", ascending=False)

In [ ]:
draw_network(G, betweenness, title="Node size = betweenness centrality")

Felix has the highest betweenness even though Gus has more friends. Most shortest paths between the left and the right group pass through Felix.

The Week 3 slides said that if such nodes are removed, the network can **fall apart**. Let's test that by removing Felix from a copy of the network:

In [ ]:
G_without_felix = G.copy()
G_without_felix.remove_node("Felix")

print("Still connected?", nx.is_connected(G_without_felix))
print("Number of separate groups:", nx.number_connected_components(G_without_felix))

draw_network(G_without_felix, title="The network without Felix")

**Task 5:** Now remove **Gus** instead of Felix (from a fresh copy of `G`). Does the network fall apart? What does this tell you about the difference between degree and betweenness?

In [ ]:
# your code here

## 9. Eigenvector centrality

**"Not what you know, but who you know."** With **eigenvector centrality**, a node is important if it is connected to nodes that are *themselves* important (Week 3). Google's PageRank works on a similar idea.

In [ ]:
eigenvector = nx.eigenvector_centrality(G)
table["eigenvector"] = pd.Series(eigenvector)
table.sort_values("eigenvector", ascending=False)

In [ ]:
draw_network(G, eigenvector, title="Node size = eigenvector centrality")

Gus has the most friends, but Dara has the highest eigenvector centrality. Many of Gus's friends (Jonas, Lena) have few friends themselves, while Dara's friends (Ana, Chen, Felix…) are well connected.

## 10. Comparing the measures

Here are all measures side by side, rounded to 2 decimals:

In [ ]:
table.round(2).sort_values("degree", ascending=False)

**Task 6 (discuss with your neighbour):** Which node has the highest **degree**, which the highest **betweenness**, and which the highest **eigenvector** centrality? Why are they different people? If you wanted to spread a piece of news to both groups quickly, whom would you tell first?

**Your answer:**

# Exercises: the karate club

Use the karate club network `karate` from Section 5. The nodes are numbered 0 to 33. Member **0** was the club's instructor and member **33** the club's administrator; a conflict between the two later split the club in two.

In [ ]:
draw_network(karate, title="Zachary's karate club")

**Exercise 1.** How many nodes and links does the network have? Calculate the density and the average degree, once with your own functions from Section 1 and once with `nx.density`. Is the network dense or sparse?

In [ ]:
# your code here

**Exercise 2.** Make a table called `karate_table` with a `degree` column (see Section 4). Which five members have the highest degree?

In [ ]:
# your code here

**Exercise 3.** What is the shortest path between member **16** and member **25**, and how long is it? What is the diameter of the network?

In [ ]:
# your code here

**Exercise 4.** Add `betweenness` and `eigenvector` columns to `karate_table`. Which three members rank highest on each measure? Are they the same members as for degree?

In [ ]:
# your code here

**Exercise 5.** Draw the network with node size by betweenness. Knowing that the club later split into two groups, what role do you think members 0 and 33 played?

**Your answer:**

In [ ]:
# your code here

# Reflection

| Measure | NetworkX | What it tells us |
|---|---|---|
| number of nodes / links | `G.number_of_nodes()`, `G.number_of_edges()` | size of the network |
| density | `nx.density(G)` | how many of the possible links exist |
| degree | `dict(G.degree())` | how many links a node has |
| degree distribution | `value_counts()` on the degree column | how degree is spread across nodes |
| shortest path / distance | `nx.shortest_path(G, a, b)`, `nx.shortest_path_length(G, a, b)` | how far apart two nodes are |
| diameter | `nx.diameter(G)` | the longest shortest path |
| betweenness centrality | `nx.betweenness_centrality(G)` | who bridges the network |
| eigenvector centrality | `nx.eigenvector_centrality(G)` | who is connected to important nodes |

## Going further (optional, not required!)

The Week 3 slides also covered two more measures. Try them on `G` or `karate`:

- **Closeness centrality:** `nx.closeness_centrality(G)`
- **Clustering coefficient:** `nx.clustering(G)`

Resources:

- NetworkX [official tutorial](https://networkx.org/documentation/stable/tutorial.html)
- List of NetworkX [centrality functions](https://networkx.org/documentation/stable/reference/algorithms/centrality.html)